# Tugas 1 — Implementasi Algoritma Apriori (Aturan Asosiasi)

**Mata Kuliah:** Data Mining

## Deskripsi
Program untuk menganalisis aturan asosiasi dari **10 transaksi** dengan **5 variasi produk**: `susu`, `gula`, `teh`, `roti`, `kopi`.

Hal yang dihitung:
1. **F1** — frekuensi setiap item (1-itemset).
2. **F2** — semua kombinasi 2-itemset. Dari 5 produk = $\binom{5}{2} = 10$ kombinasi.
3. **F3** — semua kombinasi 3-itemset. Dari 5 produk = $\binom{5}{3} = 10$ kombinasi.
4. **Support** = frekuensi itemset / total transaksi (10).
5. **Confidence** = support(A∪B) / support(A).
6. **Nilai Final** = support × confidence (untuk ranking aturan).

## 1. Import Library dan Data Transaksi

In [1]:
from itertools import combinations
import pandas as pd

# 10 transaksi dengan 5 produk: susu, gula, teh, roti, kopi
transaksi = {
    1:  {'susu', 'gula', 'teh'},
    2:  {'teh', 'gula', 'roti'},
    3:  {'susu', 'kopi', 'gula'},
    4:  {'teh', 'roti', 'kopi'},
    5:  {'susu', 'gula', 'roti'},
    6:  {'kopi', 'gula', 'teh'},
    7:  {'susu', 'teh', 'roti'},
    8:  {'kopi', 'gula', 'roti'},
    9:  {'susu', 'kopi', 'teh'},
    10: {'gula', 'roti', 'kopi'},
}

produk = ['susu', 'gula', 'teh', 'roti', 'kopi']
TOTAL_TRX = len(transaksi)

df_trx = pd.DataFrame([
    {'TID': tid, 'Items': ', '.join(sorted(items))}
    for tid, items in transaksi.items()
])
df_trx

,TID,Items
0,1,"gula, susu, teh"
1,2,"gula, roti, teh"
2,3,"gula, kopi, susu"
3,4,"kopi, roti, teh"
4,5,"gula, roti, susu"
5,6,"gula, kopi, teh"
6,7,"roti, susu, teh"
7,8,"gula, kopi, roti"
8,9,"kopi, susu, teh"
9,10,"gula, kopi, roti"


## 2. Fungsi Bantu: Hitung Support

In [2]:
def hitung_frekuensi(itemset):
    """Berapa transaksi yang memuat seluruh item dalam itemset."""
    s = set(itemset)
    return sum(1 for items in transaksi.values() if s.issubset(items))

def support(itemset):
    return hitung_frekuensi(itemset) / TOTAL_TRX

## 3. F1 — 1-itemset

In [3]:
F1 = []
for item in produk:
    f = hitung_frekuensi([item])
    F1.append({'Itemset': item, 'Frekuensi': f, 'Support': f / TOTAL_TRX})

df_F1 = pd.DataFrame(F1)
df_F1

,Itemset,Frekuensi,Support
0,susu,5,0.5
1,gula,7,0.7
2,teh,6,0.6
3,roti,6,0.6
4,kopi,6,0.6


## 4. F2 — 2-itemset (10 kombinasi)

Dari 5 produk, jumlah kombinasi 2-itemset = $\binom{5}{2} = 10$. Slide kelas hanya menampilkan 6, jadi di sini sengaja **dihitung lengkap**.

In [4]:
F2 = []
for combo in combinations(produk, 2):
    f = hitung_frekuensi(combo)
    F2.append({
        'Itemset': ' & '.join(combo),
        'Frekuensi': f,
        'Support': f / TOTAL_TRX,
    })

df_F2 = pd.DataFrame(F2)
print(f'Jumlah kombinasi F2: {len(df_F2)}  (seharusnya 10)')
df_F2

Jumlah kombinasi F2: 10  (seharusnya 10)


,Itemset,Frekuensi,Support
0,susu & gula,3,0.3
1,susu & teh,3,0.3
2,susu & roti,2,0.2
3,susu & kopi,2,0.2
4,gula & teh,3,0.3
5,gula & roti,4,0.4
6,gula & kopi,4,0.4
7,teh & roti,3,0.3
8,teh & kopi,3,0.3
9,roti & kopi,3,0.3


## 5. F3 — 3-itemset (10 kombinasi)

Dari 5 produk, jumlah kombinasi 3-itemset = $\binom{5}{3} = 10$.

In [5]:
F3 = []
for combo in combinations(produk, 3):
    f = hitung_frekuensi(combo)
    F3.append({
        'Itemset': ' & '.join(combo),
        'Frekuensi': f,
        'Support': f / TOTAL_TRX,
    })

df_F3 = pd.DataFrame(F3)
print(f'Jumlah kombinasi F3: {len(df_F3)}  (seharusnya 10)')
df_F3

Jumlah kombinasi F3: 10  (seharusnya 10)


,Itemset,Frekuensi,Support
0,susu & gula & teh,1,0.1
1,susu & gula & roti,1,0.1
2,susu & gula & kopi,1,0.1
3,susu & teh & roti,1,0.1
4,susu & teh & kopi,1,0.1
5,susu & roti & kopi,0,0.0
6,gula & teh & roti,1,0.1
7,gula & teh & kopi,1,0.1
8,gula & roti & kopi,2,0.2
9,teh & roti & kopi,1,0.1


## 6. Aturan Asosiasi (Confidence & Nilai Final)

Untuk setiap itemset di F2 dan F3, dibentuk semua aturan **A → B** dimana A∪B = itemset.

- **Support(A→B)** = support(A∪B)
- **Confidence(A→B)** = support(A∪B) / support(A)
- **Final** = Support × Confidence

In [6]:
def buat_aturan(itemset):
    """Generate semua aturan A -> B dari sebuah itemset."""
    items = list(itemset)
    aturan = []
    for r in range(1, len(items)):
        for A in combinations(items, r):
            B = tuple(sorted(set(items) - set(A)))
            sup_AB = support(items)
            sup_A = support(A)
            conf = sup_AB / sup_A if sup_A > 0 else 0
            aturan.append({
                'A': ', '.join(sorted(A)),
                'B': ', '.join(B),
                'Support': sup_AB,
                'Confidence': conf,
                'Final': sup_AB * conf,
            })
    return aturan

rules = []
for combo in combinations(produk, 2):
    rules += buat_aturan(combo)
for combo in combinations(produk, 3):
    rules += buat_aturan(combo)

df_rules = pd.DataFrame(rules)
df_rules = df_rules.sort_values('Final', ascending=False).reset_index(drop=True)
df_rules

,A,B,Support,Confidence,Final
0,kopi,gula,0.4,0.666667,0.266667
1,roti,gula,0.4,0.666667,0.266667
2,gula,roti,0.4,0.571429,0.228571
3,gula,kopi,0.4,0.571429,0.228571
4,susu,gula,0.3,0.600000,0.180000
...,...,...,...,...,...
75,"roti, susu",kopi,0.0,0.000000,0.000000
76,"kopi, susu",roti,0.0,0.000000,0.000000
77,"kopi, roti",susu,0.0,0.000000,0.000000
78,susu,"kopi, roti",0.0,0.000000,0.000000


## 7. Ranking Top-10 Aturan Asosiasi

In [7]:
top10 = df_rules.head(10).copy()
top10['Aturan'] = top10['A'] + ' → ' + top10['B']
top10[['Aturan', 'Support', 'Confidence', 'Final']]

,Aturan,Support,Confidence,Final
0,kopi → gula,0.4,0.666667,0.266667
1,roti → gula,0.4,0.666667,0.266667
2,gula → roti,0.4,0.571429,0.228571
3,gula → kopi,0.4,0.571429,0.228571
4,susu → gula,0.3,0.600000,0.180000
5,susu → teh,0.3,0.600000,0.180000
6,teh → susu,0.3,0.500000,0.150000
7,teh → roti,0.3,0.500000,0.150000
8,roti → teh,0.3,0.500000,0.150000
9,teh → gula,0.3,0.500000,0.150000


## 8. Kesimpulan

- F2 menghasilkan **10 kombinasi** lengkap (sesuai $\binom{5}{2}$), bukan 6 seperti di slide.
- F3 menghasilkan **10 kombinasi** lengkap (sesuai $\binom{5}{3}$).
- Nilai **Final = Support × Confidence** dipakai untuk meranking aturan: aturan dengan Final tertinggi merupakan aturan asosiasi terkuat dari data ini.
- Aturan teratas dapat dipakai sebagai dasar rekomendasi *cross-selling* (mis. jika pelanggan beli A → kemungkinan besar membeli B).